In [ ]:
repository_filter: list[str] = []
metric: str = "cyclomaticComplexity"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/method_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0 or metric not in df.columns:
    fig = qu.empty_figure(
        f"No data available (metric='{metric}')."
    )
else:
    # Compute median per repo and sort descending
    repo_medians = (
        df.groupby("repoShort")[metric]
        .median()
        .sort_values(ascending=False)
    )
    ordered_repos = repo_medians.index.tolist()

    # Normalize medians to 0-1 for color mapping
    med_min = repo_medians.min()
    med_max = repo_medians.max()
    med_range = med_max - med_min if med_max > med_min else 1

    fig = go.Figure()
    for repo in ordered_repos:
        repo_data = df[df["repoShort"] == repo][metric]
        median_val = repo_medians[repo]
        norm = (median_val - med_min) / med_range
        # Interpolate from green (#4CAF50) to red (#EF5350)
        r = int(76 + norm * (239 - 76))
        g = int(175 + norm * (83 - 175))
        b = int(80 + norm * (80 - 80))
        color = f"rgb({r},{g},{b})"

        fig.add_trace(
            go.Violin(
                y=repo_data,
                name=repo,
                box_visible=True,
                meanline_visible=True,
                points="outliers",
                fillcolor=color,
                line_color="#333",
                opacity=0.75,
            )
        )

    metric_label = metric.replace("_", " ").title()
    fig.update_layout(
        title=f"Portfolio Quality Comparison: {metric_label}",
        xaxis_title="Repository",
        yaxis_title=metric_label,
        showlegend=False,
        height=600,
        plot_bgcolor="white",
        margin=dict(l=60, r=40, t=60, b=120),
        xaxis=dict(tickangle=-45),
    )
fig.show()